In [ ]:
import os
import pandas as pd
import s3fs
import sklearn

sklearn.__version__

os.environ['AWS_S3_ENDPOINT']

S3_ENDPOINT_URL = 'http://' + os.environ['AWS_S3_ENDPOINT']
S3_ENDPOINT_URL

fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url' : S3_ENDPOINT_URL})

fs.ls('dlecavelier-ensae')

BUCKET = 'dlecavelier-ensae'
FILE_KEY_S3 = '/readmission_avc.parquet'
FILE_PATH_S3 = BUCKET + FILE_KEY_S3

with fs.open(FILE_PATH_S3, mode = 'rb') as file_in :
    dataini = pd.read_parquet(file_in)

dataini = dataini.dropna(axis=0, subset=['id_D']).copy()

dataini['rea'] = (dataini['id_D']!="").astype('int8')

dataini.rea.value_counts()

str_cols = ['modeEntree', 'modeSortie', 'sexe']

dataini[str_cols] = dataini[str_cols].astype("object")

dataini['nbda'] = dataini['nbda'].fillna(0)

dataset = dataini[dataini['modeSortie'] != 9].drop(['id', 'id_D'], axis = 1)

OUT_PATH_S3 = BUCKET+'/dataset.parquet'

with fs.open(OUT_PATH_S3, mode = 'wb') as file_out :
    dataset.to_parquet(file_out, index = False, compression = 'snappy')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, Normalizer

dataset['sexe'].head()

SimpleImputer(strategy='most_frequent').fit_transform(dataset[['sexe']])

OneHotEncoder(sparse_output=False, drop='first').fit_transform(SimpleImputer(strategy='most_frequent').fit_transform(dataset[['sexe']]))

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

features = dataset.drop('rea', axis = 1)

label = dataset['rea']

features.dtypes

num_features = features.select_dtypes(['int32', 'float64']).columns

cat_features = features.select_dtypes(['object']).columns

cat_transformer = Pipeline(steps= [
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features)
])

preprocessor.fit_transform(features).toarray()

from sklearn.model_selection import train_test_split

# 3 splits : le premier isole Test, le second isole Val et Train
X_train_val, X_test, y_train_val, y_test = train_test_split(features, label, random_state=18, test_size=0.1)

print(X_train_val.shape, X_test.shape)
print(y_train_val.shape, y_test.shape)

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, random_state=42, test_size=0.1)

print(X_train.shape, X_val.shape)
print(y_train.shape, y_val.shape)

from sklearn.linear_model import LogisticRegression

lr = LogisticRegression()

pip_reg = Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier', lr)
])

pip_reg_fitted = pip_reg.fit(X_train, y_train)

pip_reg_fitted.predict(X_val)

pip_reg_fitted.predict_proba(X_val)[:,1]

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

print(accuracy_score(y_val, pip_reg_fitted.predict(X_val)))

print(f1_score(y_val, pip_reg_fitted.predict(X_val)))

print(roc_auc_score(y_val, pip_reg_fitted.predict_proba(X_val)[:,1]))

from sklearn.ensemble import RandomForestClassifier
from pprint import pprint

rf = RandomForestClassifier(random_state = 42)

pip_rf = Pipeline(steps=[
    ('preproc', preprocessor),
    ('classifier',rf)
])

pip_rf

rf.get_params()

param_rf = {
    'classifier__n_estimators' : [100, 200, 500],
    'classifier__max_depth' : [10, 20]
}

metric_grid = ['accuracy', 'f1', 'roc_auc']

from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

grid_rf=GridSearchCV(
    estimator = pip_rf,
    param_grid = param_rf,
    cv = cv,
    scoring='roc_auc'
)

grid_rf_fitted = grid_rf.fit(X_train_val, y_train_val)

grid_rf_fitted.cv_results_.keys()

print(grid_rf_fitted.best_params_)

print(grid_rf_fitted.best_estimator_)

print(grid_rf_fitted.best_score_)

from sklearn.svm import SVC

svm = SVC()

pip_svm = Pipeline(steps = [
    ('preproc', preprocessor),
    ('classifier', svm)
])

param_svm = {
    'classifier__kernel' : ['linear', 'rbf', 'poly', 'sigmoid'],
    'classifier__degree' : [2, 3, 4]
}

# Model selection pipeline

models = [
    # pas de lasso CV, hyper paramètre standard pour la régression linéaire... voir 1er cours pour plus...
    ('lr', lr, {'classifier__C' : [1,0]}),
    ('rf', rf, param_rf),
    ('svm', svm, param_svm)
]

preproc_list = [
    {'id' : 'basic', 'object' : preprocessor}
]

results = []

for preproc in preproc_list :
    preproc_id = preproc['id'],
    preproc_object = preproc['object']

    for model_id, model_object, param in models :
        print(f'Model {model_id} with preprocessor {preproc_id}')

        pip = Pipeline(steps= [
            ('preproc', preproc_object),
            ('classifier', model_object)
        ])

        grid = GridSearchCV(
            estimator=pip,
            cv=cv,
            param_grid=param,
            scoring='roc_auc',
            # valeur par défaut de refit à True (le modèle est ré-entrainé sur toutes les données d'entraînement après la validation croisée et le choix des meilleurs hyper-paramètres)
            # si plusieurs métriques spécifiés dans scoring il faut spécifier lequel est utilisé pour le refit
            refit = 'roc_auc'
        ).fit(X_train_val, y_train_val)
        
        results.append(
            {
                'preprocessor' : preproc_id,
                'model' : model_id,
                'best_param' : grid.best_params_,
                'best_score' : grid.best_score_,
                'best_estimator' : grid.best_estimator_,
                'final_prediction' : grid.score(X_test, y_test)
            }
        )

pd.DataFrame(results)



(1189, 10) (133, 10)
(1189,) (133,)
(1070, 10) (119, 10)
(1070,) (119,)
0.9495798319327731
0.8235294117647058
0.8207070707070707
{'classifier__max_depth': 10, 'classifier__n_estimators': 100}
Pipeline(steps=[('preproc',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['duree', 'age', 'nbActe', 'nbRum', 'nbda'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                

/opt/python/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
5 fits failed out of a total of 10.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/python/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/python/lib/python3.13/site-packages/sklearn/base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/python/lib/python3.13/site-packages/sklearn/pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **

Model rf with preprocessor ('basic',)
Model svm with preprocessor ('basic',)


,preprocessor,model,best_param,best_score,final_prediction
0,"(basic,)",lr,{'classifier__C': 1},0.866766,0.850847
1,"(basic,)",rf,"{'classifier__max_depth': 10, 'classifier__n_e...",0.868047,0.907910
2,"(basic,)",svm,"{'classifier__degree': 2, 'classifier__kernel'...",0.847256,0.920904
